# Sparse FuseMoE Technical Specification + Code Implementation

This notebook provides a runnable implementation walkthrough for sparse-only FuseMoE (Top-K Laplace routing).

## 1) Environment and Code Structure
- Scope: **sparse FuseMoE only** (no dense/full-expert routing)

- Conda env name: `MulEHR`
- Python version: `3.8`
- Core packages: `torch`, `torchvision`, `transformers`, `numpy`, `pandas`, `scikit-learn`

### Required file structure
```text
src/
  core/
    irregularity_encoder.py
    moe_fusion.py
    routers.py
    losses.py
  preprocessing/
    mimic_iv_pipeline.py
```

In [1]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

N_EXPERTS = 16
TOP_K_JOINT = 4
TOP_K_MODALITY_SPECIFIC = 4
TOP_K_DISJOINT = 2
MOE_LAYERS = 3
ATTN_EMBED_DIM = 128
EXPERT_HIDDEN_DIM = 512

np.random.seed(42)

## 2) MIMIC-IV Preprocessing for Four Modalities

In [2]:
from src.preprocessing.mimic_iv_pipeline import (
    VitalsLabsExtractor,
    CXRExtractor,
    TextExtractor,
    ECGAutoencoder,
    MIMICIVPipeline,
    )


def build_preprocessing_pipeline(selected_events=None):
    vitals = VitalsLabsExtractor(selected_events=selected_events)
    pipeline = MIMICIVPipeline(vitals_extractor=vitals)
    return pipeline


def preprocess_batch(raw_batch, train_vitals_df=None, selected_events=None):
    pipeline = build_preprocessing_pipeline(selected_events=selected_events)

    if train_vitals_df is not None:
        pipeline.vl.fit(train_vitals_df)

    return pipeline.build(raw_batch)

## 3) Irregularity Encoder (UTDE)

UTDE combines mTAND and imputation branches with gate: `g = f(E_Imp ⊕ E_mTAND)`.

In [3]:
from src.core.irregularity_encoder import (
    UTDEConfig,
    UnifiedTemporalDiscretizationEmbedding,
    )


def build_utde(gamma_bins=48, H=16, model_dim=128):
    cfg = UTDEConfig(
        gamma_bins=gamma_bins,
        time_embed_functions=H,
        attn_embed_dim=model_dim,
    )
    return UnifiedTemporalDiscretizationEmbedding(cfg)


def encode_irregular_vitals(values, times, mask, gamma_bins=48, H=16, model_dim=128):
    utde = build_utde(gamma_bins=gamma_bins, H=H, model_dim=model_dim)
    return utde.forward(values, times, mask)

## 4) Sparse MoE Fusion with Laplace Gating

Distance-based sparse routing: `TopK(-||W-x||_2)`.

In [4]:
from src.core.moe_fusion import (
    SparseMoEConfig,
    SparseFuseMoE,
    )


def build_sparse_moe(top_k=4, n_layers=3, n_experts=16, model_dim=128, hidden_dim=512):
    cfg = SparseMoEConfig(
        n_experts=n_experts,
        top_k=top_k,
        n_layers=n_layers,
        model_dim=model_dim,
        expert_hidden_dim=hidden_dim,
    )
    return SparseFuseMoE(cfg)


def run_sparse_moe(x, top_k=4):
    moe = build_sparse_moe(top_k=top_k, n_layers=MOE_LAYERS, n_experts=N_EXPERTS)
    y, routing = moe.forward_with_routing(x)
    return y, routing

## 5) Sparse Router Architectures

In [5]:
from src.core.routers import (
    JointSparseRouter,
    ModalitySpecificSparseRouter,
    DisjointSparseRouter,
    )


def build_router(router_type, modality_names=("vitals", "cxr", "text", "ecg")):
    if router_type == "joint":
        moe = build_sparse_moe(top_k=TOP_K_JOINT, n_layers=MOE_LAYERS, n_experts=N_EXPERTS)
        return JointSparseRouter(moe, modality_order=list(modality_names))

    if router_type == "modality_specific":
        shared = build_sparse_moe(top_k=TOP_K_MODALITY_SPECIFIC, n_layers=MOE_LAYERS, n_experts=N_EXPERTS)
        return ModalitySpecificSparseRouter(shared, list(modality_names))

    if router_type == "disjoint":
        pools = {
            m: build_sparse_moe(top_k=TOP_K_DISJOINT, n_layers=MOE_LAYERS, n_experts=N_EXPERTS)
            for m in modality_names
        }
        return DisjointSparseRouter(pools)

    raise ValueError("router_type must be one of: joint, modality_specific, disjoint")

## 6) Missingness Handling and Joint Loss

In [6]:
from src.core.losses import (
    MissingnessEntropyRegularizer,
    sparse_fusemoe_joint_loss,
    )


def apply_missing_indicator(x_m, is_present, Z_m):
    is_present = np.asarray(is_present, dtype=np.float32)
    if is_present.ndim == 1:
        is_present = is_present[:, None]
    missing = (is_present <= 0.0).astype(np.float32)
    return x_m + missing * Z_m[None, :]


def entropy_regularization(router_probs_bme, missing_mask_bm):
    reg = MissingnessEntropyRegularizer(
        n_modalities=missing_mask_bm.shape[1],
        model_dim=ATTN_EMBED_DIM,
        n_experts=N_EXPERTS,
    )
    return reg.entropy_loss(router_probs_bme, missing_mask_bm)


def fusemoe_loss(task_loss, entropy_loss, lambda_e=1.0):
    return sparse_fusemoe_joint_loss(task_loss, entropy_loss, lambda_entropy=lambda_e)

## 7) End-to-End Training Loop Implementation

In [7]:
def _project_to_model_dim(x, model_dim=128, seed=0):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2:
        raise ValueError("expected [B, D] array")
    rng = np.random.default_rng(seed)
    w = rng.standard_normal((x.shape[1], model_dim), dtype=np.float32) / np.sqrt(max(1, x.shape[1]))
    return x @ w


def _prepare_modality_embeddings(batch):
    values, times, mask = batch["vitals_irregular"]
    vitals_tokens = encode_irregular_vitals(values, times, mask, gamma_bins=48, H=16, model_dim=128)
    vitals_vec = vitals_tokens.mean(axis=1)

    return {
        "vitals": _project_to_model_dim(vitals_vec, 128, seed=1),
        "cxr": _project_to_model_dim(batch["cxr"], 128, seed=2),
        "text": _project_to_model_dim(batch["text"], 128, seed=3),
        "ecg": _project_to_model_dim(batch["ecg"], 128, seed=4),
    }


def _collect_router_probs(router_out):
    # Convert router-specific outputs to [B, M, E] when routing traces are available.
    if isinstance(router_out, tuple) and len(router_out) == 2:
        _, routing = router_out
    else:
        return None

    if isinstance(routing, list):
        # Joint router: list per MoE layer of [B, E].
        return np.stack(routing, axis=1).mean(axis=1, keepdims=True)

    if isinstance(routing, dict):
        # Modality-specific/disjoint: dict[name] -> list per layer [B, E].
        names = sorted(routing.keys())
        probs = []
        for name in names:
            layer_probs = np.stack(routing[name], axis=1).mean(axis=1)
            probs.append(layer_probs)
        return np.stack(probs, axis=1)

    return None


def training_step(batch, router_type="joint", lambda_e=1.0):
    modality_embeddings = _prepare_modality_embeddings(batch)
    router = build_router(router_type, modality_names=list(modality_embeddings.keys()))

    if hasattr(router, "forward_with_routing"):
        fused, routing = router.forward_with_routing(modality_embeddings)
        router_result = (fused, routing)
    else:
        fused = router.forward(modality_embeddings)
        router_result = fused

    labels = batch.get("labels")
    if labels is None:
        labels = np.zeros((fused.shape[0],), dtype=np.float32)
    labels = np.asarray(labels, dtype=np.float32).reshape(-1)

    logits = fused.mean(axis=-1)
    task_loss = np.mean((logits - labels) ** 2).astype(np.float32)

    probs_bme = _collect_router_probs(router_result)
    if probs_bme is None:
        entropy_loss = np.float32(0.0)
    else:
        present = np.asarray(batch["modality_present"], dtype=np.float32)
        missing = (present <= 0.0).astype(np.float32)
        if probs_bme.shape[1] == 1 and missing.shape[1] > 1:
            # Joint router has one expert distribution; broadcast across modalities.
            probs_bme = np.repeat(probs_bme, missing.shape[1], axis=1)
        entropy_loss = entropy_regularization(probs_bme, missing)

    total_loss = fusemoe_loss(task_loss, entropy_loss, lambda_e=lambda_e)
    return {
        "loss": total_loss,
        "task_loss": task_loss,
        "entropy_loss": entropy_loss,
        "logits": logits,
        "fused": fused,
    }

## 8) Minimal Smoke Test

Run this cell to validate that preprocessing outputs can flow through UTDE, sparse routing, and loss computation.

In [8]:
# Synthetic multimodal batch for shape-level validation.
B, T, D = 4, 20, 30
values = np.random.randn(B, T, D).astype(np.float32)
times = np.sort(np.random.rand(B, T).astype(np.float32) * 48.0, axis=1)
mask = (np.random.rand(B, T, D) > 0.3).astype(np.float32)
values = values * mask

batch = {
    "vitals_irregular": (values, times, mask),
    "cxr": np.random.randn(B, 1024).astype(np.float32),
    "text": np.random.randn(B, 768).astype(np.float32),
    "ecg": np.random.randn(B, 256).astype(np.float32),
    "modality_present": np.array(
        [
            [1, 1, 1, 1],
            [1, 0, 1, 1],
            [1, 1, 0, 1],
            [1, 1, 1, 0],
        ],
        dtype=np.float32,
    ),
    "labels": np.random.randn(B).astype(np.float32),
}

out = training_step(batch, router_type="joint", lambda_e=1.0)
{k: (float(v) if np.isscalar(v) else np.asarray(v).shape) for k, v in out.items()}

{'loss': -0.5940957069396973,
 'task_loss': 1.036999225616455,
 'entropy_loss': -1.6310949325561523,
 'logits': (4,),
 'fused': (4, 128)}

## 9) MIMIC-IV Demo Dataset Integration

Use the helper utility to load the attached MIMIC-IV demo folder and produce a `MIMICIVPipeline.build(...)` output directly consumable by FuseMoE components.

In [9]:
from pathlib import Path
import numpy as np

from src.preprocessing.mimic_demo_loader import build_demo_pipeline_output

DEMO_ROOT = ROOT / "mimic-iv-clinical-database-demo-2.2"

demo_batch = build_demo_pipeline_output(
    demo_root=DEMO_ROOT,
    top_n_events=30,
    max_seq_len=96,
    max_events_per_episode=200,
    with_labels=True,
)

values, times, mask = demo_batch["vitals_irregular"]

summary = {
    "n_samples": len(demo_batch["ids"]),
    "vitals_values": values.shape,
    "vitals_times": times.shape,
    "vitals_mask": mask.shape,
    "cxr": demo_batch["cxr"].shape,
    "text": demo_batch["text"].shape,
    "ecg": demo_batch["ecg"].shape,
    "modality_present": demo_batch["modality_present"].shape,
    "labels": None if demo_batch["labels"] is None else np.asarray(demo_batch["labels"]).shape,
}
summary

{'n_samples': 251,
 'vitals_values': (251, 55, 30),
 'vitals_times': (251, 55),
 'vitals_mask': (251, 55, 30),
 'cxr': (251, 1024),
 'text': (251, 768),
 'ecg': (251, 256),
 'modality_present': (251, 4),
 'labels': (251,)}

In [10]:
# Optional: run one full training step on real demo-derived inputs.
demo_out = training_step(demo_batch, router_type="joint", lambda_e=1.0)
{k: (float(v) if np.isscalar(v) else np.asarray(v).shape) for k, v in demo_out.items()}

{'loss': -1.4744484424591064,
 'task_loss': 0.06117728725075722,
 'entropy_loss': -1.535625696182251,
 'logits': (251,),
 'fused': (251, 128)}